In [8]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,cross_validate,StratifiedKFold
from sklearn.metrics import make_scorer,cohen_kappa_score,matthews_corrcoef,precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.pipeline import Pipeline
from scipy.stats import wilcoxon

In [9]:
data=pd.read_csv("bank-additional-full.csv",sep=';')

FileNotFoundError: [Errno 2] No such file or directory: 'bank-additional-full.csv'

In [ ]:
data=data.drop("duration",axis=1)

In [ ]:
allColumns=data.select_dtypes(include='object').columns
for col in allColumns:
  data[col]=data[col].replace('unknown',data[col].mode()[0])

In [ ]:
data['y']=data['y'].map({'yes':1,'no':0})

In [ ]:
data=pd.get_dummies(data,drop_first=True)

In [ ]:
X=data.drop('y',axis=1)
y=data['y']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
scalar=StandardScaler()
scalar.fit_transform(X_train)
scalar.transform(X_test)

In [ ]:
kappa=make_scorer(cohen_kappa_score)
mcc=make_scorer(matthews_corrcoef)
scoring={
    'accuracy':'accuracy',
    'precision':'precision',
    'recall':'recall',
    'f1':'f1',
    'roc_auc':'roc_auc',
    'kappa':kappa,
    'mcc':mcc
}

In [ ]:
smote=SMOTE(random_state=42)
adasyn=ADASYN(random_state=42)

In [ ]:
Random_Forest=RandomForestClassifier(n_estimators=50,max_depth=10,random_state=42)
Decision_Tree=DecisionTreeClassifier(min_samples_split=2,max_depth=5,random_state=42)

In [ ]:
rf_pipeline=Pipeline([
    ('sampler',smote),
    ('model',Random_Forest)
])

dt_pipeline=Pipeline([
    ('sampler',adasyn),
    ('model',Decision_Tree)
])

In [ ]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [ ]:
rf_results=cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1
)

dt_results=cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1
)

In [ ]:
print("========== RANDOM FOREST SCORES ==========")

for metric in scoring.keys():

    print(f"\n{metric.upper()} Scores:")

    print(rf_results[f'test_{metric}'])

    print(f"Mean {metric}: {np.mean(rf_results[f'test_{metric}']):.4f}")

In [ ]:
print("========== DECISION TREE SCORES ==========")

for metric in scoring.keys():

    print(f"\n{metric.upper()} Scores:")

    print(dt_results[f'test_{metric}'])

    print(f"Mean {metric}: {np.mean(rf_results[f'test_{metric}']):.4f}")

In [ ]:
wilcoxon_results = []
alpha = 0.05
print("\n========== WILCOXON TEST RESULTS ==========")
for metric in scoring.keys():
    print(f"\n===== {metric.upper()} =====")
    # Wilcoxon Test
    statistic, p_value = wilcoxon(
        rf_results[f'test_{metric}'],
        dt_results[f'test_{metric}']
    )
    rf_mean = np.mean(
        rf_results[f'test_{metric}']
    )
    dt_mean = np.mean(
        dt_results[f'test_{metric}']
    )
    # Significance Result
    if p_value < alpha:

        result = "Significant"
    else:
        result = "Not Significant"
    wilcoxon_results.append({
        'Metric': metric,
        'RF_Mean': rf_mean,
        'DT_Mean': dt_mean,
        'Statistic': statistic,
        'P_Value': p_value,
        'Result': result
    })

    print(f"RF Mean Score : {rf_mean:.4f}")
    print(f"DT Mean Score : {dt_mean:.4f}")
    print(f"Statistic     : {statistic:.4f}")
    print(f"P-Value       : {p_value:.6f}")
    print(f"Result        : {result}")


wilcoxon_df = pd.DataFrame(wilcoxon_results)
print("\n========== FINAL COMPARISON TABLE ==========")

print(wilcoxon_df)

wilcoxon_df.to_csv(
    "Wilcoxon_Model_Comparison.csv",
    index=False
)
